In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

ROOT = Path().resolve().parent.parent

DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'

print('Racine projet :', ROOT)

df = pd.read_parquet(DATA_RAW / 'upgrade0.parquet')
df_feat = df.copy()

label_map = {
    '1A': 1, '1B': 1,
    '2A': 2, '2B': 2,
    '3A': 3, '3B': 3, '3C': 3,
    '4A': 4, '4B': 4, '4C': 4,
    '5A': 5, '5B': 5,
    '6A': 6, '6B': 6,
    '7A': 7, '7AK': 7, '7B': 7,
    '8A': 8, '8AK': 8, '8B': 8
}

climate_col = 'in.ashraeieccclimatezone2004'

if climate_col in df_feat.columns:
    df_feat[climate_col] = (
        df_feat[climate_col]
        .astype(str)
        .map(label_map)
    )
# Suppression colonnes inutiles

drop_cols = [
    c for c in [
        'in.state',
        'in.county',
        'in.city',
        'in.weather_file_city'
    ]
    if c in df_feat.columns
]

df_feat.drop(columns=drop_cols, inplace=True)


# Colonnes catégorielles

one_hot_cols = [
    'in.hvac_heating_type',
    'in.hvac_cooling_type',
    'in.heating_fuel',
    'in.geometry_building_type_recs',
    'in.census_region'
]

one_hot_cols = [c for c in one_hot_cols if c in df_feat.columns]

# Conversion category = moins de RAM
for c in one_hot_cols:
    df_feat[c] = df_feat[c].astype('category')


# One-hot encoding

df_feat = pd.get_dummies(
    df_feat,
    columns=one_hot_cols,
    dummy_na=False,
    drop_first=False
)


# Gestion des NaN

nan_pct = df_feat.isna().mean()

# Colonnes avec >30% NaN
drop_nan_cols = nan_pct[nan_pct > 0.30].index.tolist()

df_feat.drop(columns=drop_nan_cols, inplace=True)

# Colonnes numériques
num_cols = df_feat.select_dtypes(include=[np.number]).columns

# Médianes
medians = df_feat[num_cols].median()

# Colonnes à remplir
fill_cols = nan_pct[
    (nan_pct > 0) & (nan_pct < 0.05)
].index.intersection(num_cols)

for c in fill_cols:
    df_feat[c] = df_feat[c].fillna(medians[c])

# Suppression lignes restantes avec NaN
df_feat.dropna(inplace=True)



# Sauvegarde

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

df_feat.to_parquet(
    DATA_PROCESSED / 'metadata_features.parquet',
    index=False
)

print("Dataset initial :", df.shape)
print("Dataset final :", df_feat.shape)

# Taille mémoire
mem_gb = df_feat.memory_usage(deep=True).sum() / 1e9
print(f"Mémoire utilisée : {mem_gb:.2f} GB")

Racine projet : \\FS-SOP\Staff-CMA\yzouarhi\Bureau\Data\FlexiMax
Dataset initial : (549971, 771)
Dataset final : (483334, 768)
Mémoire utilisée : 3.59 GB
